In [1]:
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

# Folder dataset audio (ubah sesuai dataset kamu)
DATASET_PATH = r"D:/web/cnn_clasification/archive/Data/genres_original"
OUTPUT_CSV = "features.csv"

# Fungsi ekstraksi fitur untuk satu file audio
def extract_features(file_path, label):
    y, sr = librosa.load(file_path, sr=None, mono=True)
    features = {}
    features["filename"] = os.path.basename(file_path)
    features["length"] = len(y)

    # CHROMA
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features["chroma_stft_mean"] = np.mean(chroma)
    features["chroma_stft_var"] = np.var(chroma)

    # RMS (energy)
    rms = librosa.feature.rms(y=y)
    features["rms_mean"] = np.mean(rms)
    features["rms_var"] = np.var(rms)

    # SPECTRAL FEATURES
    cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(y)
    features["spectral_centroid_mean"] = np.mean(cent)
    features["spectral_centroid_var"] = np.var(cent)
    features["spectral_bandwidth_mean"] = np.mean(spec_bw)
    features["spectral_bandwidth_var"] = np.var(spec_bw)
    features["rolloff_mean"] = np.mean(rolloff)
    features["rolloff_var"] = np.var(rolloff)
    features["zero_crossing_rate_mean"] = np.mean(zcr)
    features["zero_crossing_rate_var"] = np.var(zcr)

    # HARMONY dan PERCEPTUAL SPREAD
    harm, perc = librosa.effects.hpss(y)
    features["harmony_mean"] = np.mean(harm)
    features["harmony_var"] = np.var(harm)
    features["perceptr_mean"] = np.mean(perc)
    features["perceptr_var"] = np.var(perc)

    # TEMPO
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    features["tempo"] = tempo

    # MFCCs (1–20)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    for i in range(1, 21):
        features[f"mfcc{i}_mean"] = np.mean(mfcc[i-1])
        features[f"mfcc{i}_var"] = np.var(mfcc[i-1])

    features["label"] = label
    return features


# Loop semua folder genre
rows = []
for root, dirs, files in os.walk(DATASET_PATH):
    for f in tqdm(files, desc=f"Scanning {root}"):
        if f.lower().endswith((".wav", ".mp3", ".ogg")):
            path = os.path.join(root, f)
            label = os.path.basename(root)
            try:
                feat = extract_features(path, label)
                rows.append(feat)
            except Exception as e:
                print(f"Error: {path} -> {e}")

# Simpan ke CSV
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Fitur disimpan ke {OUTPUT_CSV}, total {len(df)} baris.")


Scanning D:/web/cnn_clasification/archive/Data/genres_original: 0it [00:00, ?it/s]
Scanning D:/web/cnn_clasification/archive/Data/genres_original\audioanjing: 100%|██████| 25/25 [03:07<00:00,  7.49s/it]
Scanning D:/web/cnn_clasification/archive/Data/genres_original\audiobabi: 100%|████████| 24/24 [02:43<00:00,  6.80s/it]
Scanning D:/web/cnn_clasification/archive/Data/genres_original\audiobajingan: 100%|████| 24/24 [02:55<00:00,  7.32s/it]
Scanning D:/web/cnn_clasification/archive/Data/genres_original\audiobangsat: 100%|█████| 20/20 [01:52<00:00,  5.62s/it]
Scanning D:/web/cnn_clasification/archive/Data/genres_original\audiogoblok: 100%|██████| 47/47 [04:39<00:00,  5.95s/it]
Scanning D:/web/cnn_clasification/archive/Data/genres_original\audiojancuk: 100%|██████| 87/87 [09:23<00:00,  6.48s/it]
Scanning D:/web/cnn_clasification/archive/Data/genres_original\audiotai: 100%|█████████| 11/11 [02:05<00:00, 11.38s/it]
Scanning D:/web/cnn_clasification/archive/Data/genres_original\audiotolol: 10


✅ Fitur disimpan ke features.csv, total 272 baris.
